In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [3]:
experiments = client.search_experiments()
for exp in experiments:
    print(f"Name: {exp.name}, ID: {exp.experiment_id}")

Name: nyc-taxi-experiment, ID: 1
Name: Default, ID: 0


In [4]:
client.create_experiment(name="my-cool-experiment")

'2'

In [8]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 6.8",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse']
)

In [9]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 4c87752629a74322853348cbe5160dba, rmse: 6.3184
run id: 17a79314135e497c90e87a03ba6ee83d, rmse: 6.3184


In [10]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [12]:
run_id="4c87752629a74322853348cbe5160dba"
model_uri=f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-xgboost")

Registered model 'nyc-taxi-xgboost' already exists. Creating a new version of this model...
2026/05/11 11:51:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 4c87752629a74322853348cbe5160dba has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-43c3f1c5b45447ee866fb8b65406b8f6 instead
Created version '1' of model 'nyc-taxi-xgboost'.


<ModelVersion: aliases=[], creation_timestamp=1778500317880, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1778500317880, metrics=None, model_id=None, name='nyc-taxi-xgboost', params=None, run_id='4c87752629a74322853348cbe5160dba', run_link=None, source='models:/m-43c3f1c5b45447ee866fb8b65406b8f6', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [15]:
model_name = "nyc-taxi-xgboost"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version},stage: {version.current_stage}")

version: 1,stage: None


/tmp/ipykernel_25375/3379697229.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [18]:
model_version = 1
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_25375/1600074043.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778500317880, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1778501095243, metrics=None, model_id=None, name='nyc-taxi-xgboost', params=None, run_id='4c87752629a74322853348cbe5160dba', run_link=None, source='models:/m-43c3f1c5b45447ee866fb8b65406b8f6', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [19]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name = model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1778500317880, current_stage='Staging', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2026-05-11', last_updated_timestamp=1778501198045, metrics=None, model_id=None, name='nyc-taxi-xgboost', params=None, run_id='4c87752629a74322853348cbe5160dba', run_link=None, source='models:/m-43c3f1c5b45447ee866fb8b65406b8f6', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [41]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [34]:
df = read_dataframe('./data/green_tripdata_2021-03.parquet')

In [23]:
client.download_artifacts(run_id=run_id, path="preprocessor", dst_path=".")

'/workspaces/mlops/2-experiment_tracking/preprocessor'

In [24]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [35]:
X_test = preprocess(df,dv)

In [36]:
target = "duration"
y_test = df[target].values

In [42]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 17.1 s, sys: 50.9 ms, total: 17.1 s
Wall time: 9.94 s


{'rmse': 6.2702965482607915}

In [43]:
client.transition_model_version_stage(
    name=model_name,
    version=1,
    stage="Production",
    archive_existing_versions=True
)

/tmp/ipykernel_25375/1316468422.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778500317880, current_stage='Production', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2026-05-11', last_updated_timestamp=1778502410267, metrics=None, model_id=None, name='nyc-taxi-xgboost', params=None, run_id='4c87752629a74322853348cbe5160dba', run_link=None, source='models:/m-43c3f1c5b45447ee866fb8b65406b8f6', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>